In [ ]:
# ==========================================================
# MINERÍA DE DATOS + APRENDIZAJE SUPERVISADO (SCRIPT ÚNICO)
# Notebook listo para Google Colab
#
# Este script incluye:
# - Contexto del ejercicio (qué problema resuelve y por qué)
# - Descripción del dataset (origen, tamaño, variables, objetivo)
# - EDA básico (comprensión del dato)
# - Ejemplo de calidad de datos (simulación de faltantes + imputación)
# - Baseline (referencia mínima)
# - Pipeline (imputación + escalado + modelo)
# - Evaluación (accuracy, AUC, reporte, matriz de confusión, ROC)
# - Validación cruzada (estabilidad)
# - Interpretación (variables influyentes por coeficientes)
# ==========================================================

## 1. Contexto y objetivo

En aprendizaje supervisado conocemos ejemplos históricos con su respuesta correcta. Aquí, cada fila representa una muestra y la variable objetivo indica si fue clasificada como maligna (0) o benigna (1).

El objetivo no es sólo obtener un porcentaje alto: también debemos comprobar que el modelo generaliza a datos no vistos, entender sus errores y revisar si su rendimiento es estable.

In [ ]:
# -----------------------------
# 0) CONTEXTO DEL EJERCICIO
# -----------------------------

# Imprimimos un título para que el notebook sea autoexplicativo al ejecutarse.
print("=== Práctica: Minería de Datos con Aprendizaje Supervisado ===")

# Explicamos el objetivo general de la práctica en texto visible (salida estándar).
print(
    "\nObjetivo: construir un modelo de clasificación (aprendizaje supervisado) "
    "que prediga si un tumor es benigno o maligno a partir de mediciones numéricas.\n"
)

# Enumeramos las fases de minería de datos que se ejecutan en este notebook.
print(
    "Etapas incluidas:\n"
    "1) Comprensión del problema\n"
    "2) Comprensión del dataset (EDA)\n"
    "3) Calidad y preparación de datos\n"
    "4) Baseline (referencia mínima)\n"
    "5) Modelado con Pipeline (sin fuga de información)\n"
    "6) Evaluación con métricas y visualizaciones\n"
    "7) Validación cruzada\n"
    "8) Interpretación del modelo\n"
)

## 2. Librerías

Se cargan las herramientas necesarias para manipular tablas (`pandas`), realizar cálculos (`numpy`), entrenar el modelo y medir su desempeño. Separar las librerías por función facilita entender qué parte del flujo realiza cada tarea.

In [ ]:
# -----------------------------
# 1) IMPORTACIÓN DE LIBRERÍAS
# -----------------------------

# Importamos numpy (np) para operaciones numéricas y generación controlada de aleatoriedad.
import numpy as np

# Importamos pandas (pd) para manipular los datos en forma tabular (DataFrame/Series).
import pandas as pd


In [ ]:
# Importamos el dataset integrado "breast cancer" de scikit-learn.
from sklearn.datasets import load_breast_cancer

In [ ]:
# Importamos la función para dividir datos en entrenamiento y prueba.
from sklearn.model_selection import train_test_split


In [ ]:
# Importamos cross_val_score para validación cruzada (k-fold).
from sklearn.model_selection import cross_val_score

In [ ]:
# Importamos DummyClassifier para construir un baseline sencillo.
from sklearn.dummy import DummyClassifier


In [ ]:
# Importamos Pipeline para encadenar pasos y evitar fuga de información (data leakage).
from sklearn.pipeline import Pipeline


In [ ]:
# Importamos StandardScaler para estandarizar variables numéricas.
from sklearn.preprocessing import StandardScaler

In [ ]:
# Importamos SimpleImputer para imputar valores faltantes.
from sklearn.impute import SimpleImputer

In [ ]:
# Importamos LogisticRegression como modelo supervisado (interpretabilidad por coeficientes).
from sklearn.linear_model import LogisticRegression

In [ ]:
# Importamos métricas de evaluación: accuracy, reporte, matriz, AUC, ROC.
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

In [ ]:
# Importamos matplotlib para gráficas.
import matplotlib.pyplot as plt

## 3. Carga y descripción de los datos

El dataset se obtiene directamente desde scikit-learn, por lo que el notebook es reproducible en Google Colab. `X` contiene las 30 mediciones numéricas y `y` contiene la respuesta que queremos predecir.

In [ ]:
# -----------------------------------------
# 2) CARGA Y DESCRIPCIÓN DEL DATASET (INCLUIDO)
# -----------------------------------------

# Cargamos el dataset Breast Cancer Wisconsin (Diagnostic) como objeto tipo "Bunch".
data = load_breast_cancer()

In [ ]:
# Convertimos la matriz de características a un DataFrame con nombres de columnas.
X = pd.DataFrame(data.data, columns=data.feature_names)


In [ ]:
# Convertimos el vector objetivo a una Serie con nombre "target".
y = pd.Series(data.target, name="target")


In [ ]:
# Guardamos los nombres de las clases para reportes legibles.
target_names = data.target_names  # típicamente: ['malignant', 'benign']

In [ ]:
# Imprimimos un resumen del dataset para documentar tamaño y objetivo.
print("\n=== Dataset: Breast Cancer Wisconsin (Diagnostic) ===")
print("Instancias (filas):", X.shape[0])          # número de observaciones
print("Variables (columnas):", X.shape[1])        # número de features
print("Clases disponibles:", list(target_names))  # etiquetas humanas
print("Nota: en este dataset, 0 = malignant, 1 = benign (según scikit-learn).")

## 4. Exploración inicial (EDA)

Antes de entrenar, revisamos forma, tipos de datos, valores faltantes, distribución de clases y escalas. Esta revisión ayuda a detectar problemas que podrían afectar el modelo y permite interpretar mejor los resultados posteriores.

In [ ]:
# -----------------------------
# 3) EDA BÁSICO (COMPRENDER EL DATO)
# -----------------------------

# Mostramos las primeras filas para tener un vistazo del formato de los datos.
print("\n=== Primeras 5 filas de X ===")
print(X.head())

In [ ]:
# Mostramos los tipos de dato por columna (en este dataset, deberían ser numéricos).
print("\n=== Tipos de dato (conteo) ===")
print(X.dtypes.value_counts())

In [ ]:
# Revisamos valores faltantes reales (en este dataset normalmente no hay).
print("\n=== Valores faltantes reales (top 10 columnas) ===")
print(X.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Revisamos balance/desbalance de clases (minería de datos: distribución del objetivo).
print("\n=== Distribución de clases ===")
print(y.value_counts().rename(index={0: "malignant(0)", 1: "benign(1)"}))

In [ ]:
# Resumen estadístico para comprender escalas y dispersión (importante para preprocesamiento).
print("\n=== Estadística descriptiva (primeras 8 variables) ===")
print(X.describe().T.head(8))

## 5. Calidad de datos: faltantes simulados

El dataset original está limpio. Para demostrar una situación común en proyectos reales, se introduce de forma controlada un 1% de valores faltantes. Después, el pipeline los reemplazará usando la mediana calculada sólo con el conjunto de entrenamiento.

In [ ]:
# ------------------------------------------------------
# 4) CALIDAD DE DATOS: SIMULACIÓN CONTROLADA DE FALTANTES
# ------------------------------------------------------
# En problemas reales es común encontrar valores faltantes.
# Este dataset viene limpio; por ello, simularemos un pequeño % de NaN
# para ejemplificar imputación dentro del pipeline (práctica típica de MD).

# Creamos una copia para no modificar el DataFrame original.
X_q = X.copy()

In [ ]:
# Creamos un generador aleatorio reproducible con semilla fija.
rng = np.random.default_rng(42)

In [ ]:
# Definimos el porcentaje de celdas que convertiremos a NaN.
missing_rate = 0.01  # 1% de celdas faltantes simuladas

In [ ]:
# Calculamos el número total de celdas (filas * columnas).
n_cells = X_q.shape[0] * X_q.shape[1]

In [ ]:
# Calculamos cuántas celdas serán NaN según missing_rate.
n_missing = int(n_cells * missing_rate)

In [ ]:
# Elegimos índices lineales únicos en [0, n_cells) para colocar NaN.
missing_indices = rng.choice(n_cells, size=n_missing, replace=False)

In [ ]:
# Convertimos índices lineales a (fila, columna).
rows = missing_indices // X_q.shape[1]
cols = missing_indices % X_q.shape[1]


In [ ]:
# Asignamos NaN en las posiciones seleccionadas.
X_q.values[rows, cols] = np.nan

In [ ]:
# Verificamos cuántos NaN quedaron por columna (solo para inspección).
print("\n=== Valores faltantes simulados (top 10 columnas) ===")
print(X_q.isna().sum().sort_values(ascending=False).head(10))


## 6. Separación entre entrenamiento y prueba

El 80% de los datos se utiliza para aprender y el 20% se reserva para una evaluación imparcial. `stratify=y` conserva la proporción de tumores malignos y benignos en ambos grupos; `random_state=42` hace que el resultado pueda repetirse.

In [ ]:
# -----------------------------
# 5) TRAIN/TEST SPLIT
# -----------------------------
# Objetivo: evaluar rendimiento en datos no vistos.
# - test_size=0.2 => 20% para prueba
# - stratify=y => mantiene proporción de clases
# - random_state=42 => reproducibilidad

# Ejecutamos la separación.
X_train, X_test, y_train, y_test = train_test_split(
    X_q, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# Mostramos tamaños para confirmar partición.
print("\n=== Split Train/Test ===")
print("Train:", X_train.shape, "Test:", X_test.shape)


## 7. Baseline: una referencia mínima

El baseline siempre predice la clase más frecuente. No es un modelo inteligente; sirve como punto de comparación. Si el modelo supervisado no supera claramente esta referencia, sus predicciones no aportarían valor práctico.

In [ ]:
# -----------------------------
# 6) BASELINE (REFERENCIA MÍNIMA)
# -----------------------------
# Un baseline ayuda a evitar autoengaño: si el modelo no supera esto, no aporta valor.

# Creamos un clasificador que predice siempre la clase más frecuente del training.
baseline = DummyClassifier(strategy="most_frequent", random_state=42)


In [ ]:
# Entrenamos el baseline con los datos de entrenamiento.
baseline.fit(X_train, y_train)

In [ ]:
# Predecimos sobre el conjunto de prueba.
y_pred_base = baseline.predict(X_test)


In [ ]:
# Calculamos el accuracy del baseline.
acc_base = accuracy_score(y_test, y_pred_base)

In [ ]:
# Reportamos el desempeño del baseline.
print("\n=== Baseline ===")
print("Accuracy baseline (clase más frecuente):", acc_base)


## 8. Preparación y entrenamiento del modelo

El pipeline encadena tres pasos: (1) completar faltantes con la mediana, (2) estandarizar las variables para ponerlas en escalas comparables y (3) entrenar una regresión logística.

La ventaja clave es evitar la **fuga de información**: el imputador y el escalador aprenden sus parámetros únicamente con `X_train` y luego se aplican a `X_test`.

In [ ]:
# -------------------------------------------------------
# 7) PIPELINE: IMPUTACIÓN + ESCALADO + MODELO
# -------------------------------------------------------
# Pipeline evita data leakage: el imputador y el escalador se ajustan SOLO con train
# y luego se aplican a test con los parámetros aprendidos en train.

# Definimos el pipeline con tres pasos.
pipe = Pipeline(steps=[
    # Imputación por mediana (robusta a outliers).
    ("imputer", SimpleImputer(strategy="median")),
    # Estandarización: media 0, desviación 1.
    ("scaler", StandardScaler()),
    # Modelo: regresión logística (interpretación por coeficientes).
    ("clf", LogisticRegression(max_iter=5000, solver="lbfgs"))
])

In [ ]:
# Entrenamos el pipeline completo.
pipe.fit(X_train, y_train)

In [ ]:
# Predecimos clases en test.
y_pred = pipe.predict(X_test)

In [ ]:
# Predecimos probabilidades de la clase positiva (1=benign) para AUC/ROC.
y_proba = pipe.predict_proba(X_test)[:, 1]

## 9. Evaluación del modelo

Se calculan varias métricas porque una sola puede ocultar errores importantes. `accuracy` indica la proporción total de aciertos; precisión, recall y F1 permiten estudiar cada clase; ROC-AUC resume qué tan bien separa ambas clases considerando distintos umbrales.

In [ ]:
# -----------------------------
# 8) EVALUACIÓN DEL MODELO
# -----------------------------

# Calculamos accuracy del modelo entrenado.
acc = accuracy_score(y_test, y_pred)

In [ ]:
# Calculamos AUC ROC (mide capacidad de separar clases, independiente del umbral).
auc = roc_auc_score(y_test, y_proba)

In [ ]:
# Imprimimos métricas principales.
print("\n=== Modelo Supervisado (Pipeline + LogisticRegression) ===")
print("Accuracy:", acc)
print("ROC AUC:", auc)

### Cómo leer estas métricas

Compara el resultado con el baseline. Un accuracy cercano a 1 significa que la mayoría de las predicciones fueron correctas, pero siempre conviene revisar también los errores por clase. Un ROC-AUC cercano a 1 indica que el modelo ordena muy bien los casos de una clase frente a la otra.

En un contexto médico, el **recall de malignos** es especialmente importante: representa la proporción de tumores malignos que el modelo logra detectar.

In [ ]:
# Reporte con precision, recall y f1 por clase.
print("\n=== Classification Report ===")
print(classification_report(
    y_test, y_pred,
    target_names=[f"{target_names[0]}(0)", f"{target_names[1]}(1)"]
))

In [ ]:
# Matriz de confusión para observar errores por clase.
cm = confusion_matrix(y_test, y_pred)


In [ ]:
# Mostramos matriz en texto.
print("=== Matriz de Confusión ===")
print(cm)

In [ ]:
print("\n=== Interpretación sencilla de la matriz ===")
tn, fp, fn, tp = cm.ravel()
print(f"Malignos correctamente detectados: {tn} de {tn + fp}.")
print(f"Malignos confundidos como benignos: {fn}.")
print(f"Benignos correctamente detectados: {tp} de {tp + fn}.")
print("Un falso negativo es el error más delicado en este tipo de problema: un caso maligno clasificado como benigno." )

In [ ]:
# Graficamos matriz de confusión.
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Matriz de Confusión (Breast Cancer)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks([0, 1], [f"{target_names[0]}(0)", f"{target_names[1]}(1)"])
plt.yticks([0, 1], [f"{target_names[0]}(0)", f"{target_names[1]}(1)"])
plt.colorbar()
plt.show()

## 10. Curva ROC

La curva ROC muestra el equilibrio entre detectar correctamente la clase positiva y generar falsos positivos. Cuanto más se acerque la curva a la esquina superior izquierda, mejor separa el modelo. La diagonal representa un clasificador equivalente al azar.

In [ ]:
# -----------------------------
# 9) CURVA ROC
# -----------------------------

# Calculamos el FPR, TPR y umbrales para la curva ROC.
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

In [ ]:
# Graficamos la curva ROC.
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")  # referencia: clasificador aleatorio
plt.title("Curva ROC")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.show()


## 11. Validación cruzada

La validación cruzada divide repetidamente los datos en cinco partes. Cada parte funciona una vez como validación y las otras cuatro como entrenamiento. El promedio resume el rendimiento y la desviación estándar indica cuánto cambia entre particiones.

In [ ]:
# ---------------------------------
# 10) VALIDACIÓN CRUZADA (K-FOLD)
# ---------------------------------
# Sirve para estimar desempeño promedio y variabilidad sin depender de un solo split.

# Ejecutamos validación cruzada 5-fold con accuracy.
cv_scores = cross_val_score(pipe, X_q, y, cv=5, scoring="accuracy")

In [ ]:
# Reportamos resultados por fold y resumen.
print("\n=== Validación Cruzada (5-fold) ===")
print("Accuracies por fold:", cv_scores)
print("Promedio:", cv_scores.mean())
print("Desviación estándar:", cv_scores.std())

### Interpretación de la estabilidad

Un promedio alto con una desviación estándar pequeña indica que el resultado no depende demasiado de una única partición. Aun así, la validación cruzada no reemplaza una validación externa con datos nuevos de otra fuente.

## 12. Interpretación de variables

La regresión logística asigna un coeficiente a cada variable. Como los datos fueron estandarizados, la magnitud absoluta permite comparar la influencia relativa entre variables. Un coeficiente positivo empuja la predicción hacia la clase 1 (benigna) y uno negativo hacia la clase 0 (maligna), manteniendo constantes las demás variables.

Esto describe asociaciones dentro de este dataset; no demuestra causalidad médica.

In [ ]:
# ------------------------------------------------
# 11) INTERPRETACIÓN: COEFICIENTES DEL MODELO
# ------------------------------------------------
# En regresión logística, el valor del coeficiente indica influencia:
# - signo (+/-) sugiere dirección de la asociación
# - magnitud (|coef|) sugiere importancia relativa (tras escalado)

# Extraemos el clasificador entrenado del pipeline.
clf = pipe.named_steps["clf"]

In [ ]:

# Extraemos el vector de coeficientes (una fila para clasificación binaria).
coefs = clf.coef_.ravel()

In [ ]:
# Construimos una tabla para ordenar por magnitud absoluta.
importance = pd.DataFrame({
    "feature": X.columns,
    "coef": coefs,
    "abs_coef": np.abs(coefs)
}).sort_values("abs_coef", ascending=False)


In [ ]:
# Imprimimos top 10 variables más influyentes.
print("\n=== Top 10 variables más influyentes (|coef|) ===")
print(importance.head(10)[["feature", "coef"]])


### Interpretación de los coeficientes

Las variables mostradas son las que más contribuyen a separar las clases dentro del modelo entrenado. El signo indica dirección y el tamaño indica fuerza relativa después del escalado. No debe interpretarse como una regla clínica independiente ni como evidencia de que una variable cause la enfermedad.

In [ ]:
# Graficamos top 10 por magnitud absoluta.
plt.figure(figsize=(8, 4))
plt.bar(importance["feature"].head(10), importance["abs_coef"].head(10))
plt.title("Top 10 Importancia por |Coeficiente| (LogReg)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("|coef|")
plt.show()

## 13. Conclusiones

Este bloque final resume los hallazgos observados y traduce las métricas a una lectura sencilla. Se ejecuta al final para garantizar que las conclusiones correspondan a los resultados obtenidos en la corrida actual.

In [ ]:
print("=== Conclusiones del análisis ===")
print(f"1. El dataset contiene {X.shape[0]} casos y {X.shape[1]} variables numéricas.")
print(f"2. La clase mayoritaria es benigna; el baseline alcanzó {acc_base:.2%} de accuracy.")
print(f"3. La regresión logística alcanzó {acc:.2%} de accuracy y un ROC-AUC de {auc:.4f} en el conjunto de prueba.")
print(f"4. Detectó {tn} de {tn + fp} casos malignos y dejó {fn} falso(s) negativo(s) en esta partición.")
print(f"5. En validación cruzada, el accuracy promedio fue {cv_scores.mean():.2%} con desviación estándar de {cv_scores.std():.4f}.")
print("6. El pipeline permitió tratar faltantes y escalar variables sin fuga de información.")
print("7. El desempeño es muy bueno para este dataset académico, pero antes de cualquier uso real se necesitarían datos externos, validación clínica, análisis de sesgos y revisión de costos de error.")